# Figure Generation — PokerBeliefStudy

This notebook generates all publication-ready figures at 300 DPI.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.analysis import (
    compute_performance_summary,
    compute_calibration_metrics,
    compute_robustness_metrics,
    plot_performance_comparison,
    plot_reliability_diagram,
    plot_robustness_heatmap,
    plot_belief_trace,
    generate_all_tables,
    generate_all_figures,
)

# Directories
OUTPUT_DIR = '../outputs'
RAW_DIR = os.path.join(OUTPUT_DIR, 'raw_runs')
PROCESSED_DIR = os.path.join(OUTPUT_DIR, 'processed')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
TABLES_DIR = os.path.join(OUTPUT_DIR, 'tables')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

print('Setup complete.')
print(f'Figures will be saved to: {os.path.abspath(FIGURES_DIR)}')

## Generate All Figures Automatically

In [ ]:
# Auto-generate all figures from available data
try:
    generate_all_figures(PROCESSED_DIR, RAW_DIR, FIGURES_DIR)
    print('\nAll figures generated successfully.')
except Exception as e:
    print(f'Error generating figures: {e}')
    print('Make sure you have run at least one experiment first.')

## Figure 1: Performance Comparison

In [ ]:
summary_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('_summary.csv')]

for fname in summary_files:
    exp_id = fname.replace('_summary.csv', '')
    df = pd.read_csv(os.path.join(PROCESSED_DIR, fname))
    
    if 'agent0_type' not in df.columns:
        continue
    
    summary = compute_performance_summary(df)
    if summary.empty:
        continue
    
    out_path = os.path.join(FIGURES_DIR, f'{exp_id}_performance.png')
    plot_performance_comparison(summary, out_path)
    
    # Display inline
    img = plt.imread(out_path)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Figure 1: Performance Comparison — {exp_id}', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Saved: {out_path}')

## Figure 2: Calibration Reliability Diagram

In [ ]:
# Load raw records for calibration analysis
raw_records = []
raw_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.json')]

for fname in raw_files[:20]:
    with open(os.path.join(RAW_DIR, fname)) as f:
        batch = json.load(f)
    raw_records.extend(batch)

print(f'Loaded {len(raw_records):,} hand records')

if raw_records:
    cal_data = compute_calibration_metrics(raw_records)
    out_path = os.path.join(FIGURES_DIR, 'calibration_reliability.png')
    plot_reliability_diagram(cal_data, out_path)
    
    img = plt.imread(out_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Figure 2: Calibration Reliability Diagram', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Saved: {out_path}')
    
    print(f'\nCalibration metrics:')
    print(f'  Brier Score: {cal_data["brier_score"]}')
    print(f'  ECE: {cal_data["ece"]}')
else:
    print('No raw records. Run an experiment first.')

## Figure 3: Robustness Heatmap

In [ ]:
robustness_files = [f for f in os.listdir(PROCESSED_DIR) 
                    if 'robustness' in f and f.endswith('_summary.csv')]

for fname in robustness_files:
    exp_id = fname.replace('_summary.csv', '')
    df = pd.read_csv(os.path.join(PROCESSED_DIR, fname))
    
    robustness = compute_robustness_metrics(df)
    if robustness.empty:
        continue
    
    out_path = os.path.join(FIGURES_DIR, f'{exp_id}_robustness.png')
    plot_robustness_heatmap(robustness, out_path)
    
    img = plt.imread(out_path)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Figure 3: Robustness Heatmap — {exp_id}', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Saved: {out_path}')

## Figure 4: Belief Trace Example

In [ ]:
# Find a hand with belief data
belief_hand = None
for rec in raw_records:
    has_belief = any(
        step.get('posterior') is not None
        for step in rec.get('action_history', [])
    )
    if has_belief:
        belief_hand = rec
        break

if belief_hand:
    out_path = os.path.join(FIGURES_DIR, 'belief_trace_example.png')
    plot_belief_trace(belief_hand, out_path)
    
    img = plt.imread(out_path)
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Figure 4: Belief Trace Example', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Saved: {out_path}')
else:
    print('No belief hands found. Run belief_ablation or calibration experiment.')

## Generate All Tables

In [ ]:
try:
    generate_all_tables(PROCESSED_DIR, TABLES_DIR)
    
    # List generated tables
    table_files = os.listdir(TABLES_DIR)
    print(f'Generated {len(table_files)} table files:')
    for f in sorted(table_files):
        fpath = os.path.join(TABLES_DIR, f)
        size = os.path.getsize(fpath)
        print(f'  {f} ({size} bytes)')
except Exception as e:
    print(f'Error generating tables: {e}')

## Summary

In [ ]:
print('=== Figure Generation Summary ===')

# List all generated figures
fig_files = [f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')]
print(f'\nFigures ({len(fig_files)} total):')
for f in sorted(fig_files):
    fpath = os.path.join(FIGURES_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f} ({size_kb:.0f} KB)')

# List all generated tables
tbl_files = [f for f in os.listdir(TABLES_DIR) if f.endswith(('.csv', '.tex'))]
print(f'\nTables ({len(tbl_files)} total):')
for f in sorted(tbl_files):
    print(f'  {f}')

print(f'\nAll outputs saved to: {os.path.abspath(OUTPUT_DIR)}')